# 🩺 Kiểm Tra & Đánh Giá Tiền Xử Lý Dữ Liệu (Mentor Day 1)

Notebook này hỗ trợ kiểm tra chi tiết từng bước ở môi trường local trước khi mở rộng ra toàn bộ kho:
1. **Bóc tách cấu trúc theo 4 Họ tài liệu (Document Archetypes)**: Q&A (`120ask`), Bài kiến thức (`Long Châu`, `Sức Khỏe Đời Sống`), Tin tức y tế (`Thanh Niên`), và PubMed tiếng Anh.
2. **Làm sạch nhiễu & Bảo tồn cấu trúc**: Giữ nguyên H2/H3, loại bỏ hoàn toàn hotline, quảng cáo, và phần giới thiệu dược sĩ/bác sĩ.
3. **Phân loại chủ đề ICD-10 tự động**: Ánh xạ đa ngữ (Việt - Anh - Trung) về 22 Chương bệnh ICD-10 của WHO.
4. **Kiểm tra Chunking theo cấu trúc**: Đảm bảo các chunk không bị cắt ngang câu và giữ ngữ cảnh.

In [ ]:
# 1. Cấu hình môi trường & Import modules
import sys
from pathlib import Path
import pandas as pd
import httpx
import json
from IPython.display import display, Markdown, HTML

# Đảm bảo nạp đúng mã nguồn dự án
REPO_ROOT = Path(".." if Path(".").resolve().name == "notebooks" else ".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.crawler.extractors import get_extractor, extract_article
from src.ingestion.icd10_tagger import ICD10Tagger
from src.ingestion.chunker import DocumentChunker

tagger = ICD10Tagger()
chunker = DocumentChunker(max_chunk_size=512, chunk_overlap=64, min_chunk_size=50)
print("✓ Khởi tạo modules thành công!")

## 2. Danh Sách URL Mẫu Đại Diện Cho 4 Họ Chiến Thuật

In [ ]:
sample_urls = [
    # Họ 1: Hỏi đáp y khoa (Q&A)
    {"name": "120ask (Hỏi đáp TQ)", "archetype": "qa", "url": "http://www.120ask.com/question/103123331.htm"},
    # Họ 2: Bách khoa bệnh học & Dược
    {"name": "Nhà thuốc Long Châu", "archetype": "disease_encyclopedia", "url": "https://nhathuoclongchau.com.vn/bai-viet/1-banh-trung-thu-bang-may-bat-com-goc-nhin-dinh-duong-va-suc-khoe.html"},
    {"name": "Sức Khỏe Đời Sống", "archetype": "disease_encyclopedia", "url": "https://suckhoedoisong.vn/6-loai-do-uong-tang-cuong-trao-doi-chat-dot-chay-calo-hieu-qua-1692505311408571.htm"},
    # Họ 3: Báo chí & Tin tức y tế
    {"name": "Báo Thanh Niên", "archetype": "news", "url": "https://thanhnien.vn/an-qua-nhieu-duong-co-hai-gi-cho-suc-khoe-185764864.htm"},
]
pd.DataFrame(sample_urls)

## 3. Tải & Trích Xuất Cấu Trúc Kèm Phân Loại ICD-10

In [ ]:
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
results = []

with httpx.Client(headers=headers, timeout=12.0, follow_redirects=True) as client:
    for item in sample_urls:
        url = item["url"]
        try:
            r = client.get(url)
            raw_html = r.text
            extracted = extract_article(raw_html, url)
            if extracted:
                # Gán nhãn ICD-10 tự động
                icd_res = tagger.tag_document(
                    text=extracted["text"],
                    title=extracted.get("title", ""),
                    headings=extracted.get("headings", []),
                )
                results.append({
                    "Source": item["name"],
                    "Archetype": extracted.get("doc_type"),
                    "Title": extracted.get("title", "")[:50],
                    "Raw Bytes": len(r.content),
                    "Text Chars": len(extracted["text"]),
                    "Headings": len(extracted.get("headings", [])),
                    "QA Pairs": len(extracted.get("qa_pairs", [])),
                    "ICD-10 Chapter": icd_res.primary_chapter_code or "N/A",
                    "ICD Terms": ", ".join(icd_res.matched_terms[:3]),
                    "_full": extracted,
                    "_icd": icd_res,
                })
        except Exception as e:
            print(f"Lỗi tải {url}: {e}")

summary_df = pd.DataFrame([{
    k: v for k, v in r.items() if not k.startswith("_")
} for r in results])
display(summary_df)

## 4. Xem Trực Quan Bản Markdown Bóc Tách (Preview Nội Dung Sạch)

In [ ]:
# Chọn bài viết để xem nội dung (0: 120ask Q&A, 1: Long Châu, 2: Sức Khỏe Đời Sống)
idx_to_view = 1
item = results[idx_to_view]

display(HTML(f"<h3>Nguồn: {item['Source']} ({item['Archetype']})</h3>"))
print(f"Chương ICD-10: {item['_icd'].primary_chapter}")
print(f"Các mã ICD phát hiện: {item['_icd'].matched_codes}")
print(f"Thuật ngữ y khoa phát hiện: {item['_icd'].matched_terms}")
print("-" * 80)

# Hiển thị 1000 ký tự đầu tiên dạng Markdown
snippet = item["_full"]["text"][:1000] + "\n\n*(Còn tiếp...)*"
display(Markdown(snippet))

## 5. Thử Nghiệm Chunking Theo Cấu Trúc (Heading Preserved)

In [ ]:
doc_text = item["_full"]["text"]
doc_title = item["_full"]["title"]

chunks = chunker.chunk_document(
    doc_id="test_doc_001",
    text=doc_text,
    metadata={"title": doc_title},
)

print(f"Tài liệu được chia thành {len(chunks)} chunks.")
for i, c in enumerate(chunks[:3]):
    print(f"\n--- CHUNK {i} ({len(c.chunk_text)} ký tự, spans: [{c.char_start}:{c.char_end}]) ---")
    print(c.chunk_text[:250] + "...")
    if c.contextual_text:
        print(f"[Contextual Text Preview]:\n{c.contextual_text[:120]}...")

## 6. Chạy Mini-Batch trên 30 URL thực tế từ missing_urls.parquet

In [ ]:
import pyarrow.parquet as pq

missing_path = REPO_ROOT / "data/processed/v2/missing_urls.parquet"
if missing_path.exists():
    tbl = pq.read_table(missing_path)
    df_missing = tbl.to_pandas()
    top_hosts = ["nhathuoclongchau.com.vn", "suckhoedoisong.vn", "www.120ask.com", "thanhnien.vn", "www.vinmec.com"]
    df_sample = df_missing[df_missing["host"].isin(top_hosts)].groupby("host").head(6)
    print(f"Đã chọn {len(df_sample)} URL mẫu từ 5 host chính để test batch.")
    display(df_sample[["id", "host", "url"]].head(10))
else:
    print("Chưa tìm thấy missing_urls.parquet. Hãy chạy scripts/audit_coverage.py trước.")